# DL4CV PA02-Q6: Convolutional Neural Networks and Introduction to PyTorch


---

### $\color{red}{\textbf{Name and ID:}}$ XXXXXXXX YourName(Fill here)

Before we start, please put your student ID and your name in following format.
For example

12345678 HONGGildong


---

### Submission

1. The output of the code cells in this `.ipynb` file $\color{red}{\text{MUST}}$ be saved.
   - Save text output of code cells
   - Save graphs and plots of code cells    
2. All The `.py` files should be submitted.
3. zip the required `.ipynb` and `.py` files into $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ and submit.
   - For example `12345678_HONGGildong.zip`
4. Deadline is $\color{red}{2026:10:17-23:59:59}$ (YYYY:MM:DD-HH:MM:SS)

---

### To-do

1. Fill in Your ID and Name.
2. (Programming) In `CNN.py`, implement **conv_forward()** function.
3. (Programming) In `CNN.py`, implement **conv_backward()** function.
4. (Programming) In `CNN.py`, implement **max_pool_forward()** function.
5. (Programming) In `CNN.py`, implement **max_pool_backward()** function.
6. (Programming) In `three_layer_conv_net.py`, implement **`__init__()`** function in `class ThreeLayerConvNet`.
7. (Programming) In `three_layer_conv_net.py`, implement **loss()** function in `class ThreeLayerConvNet`.
8. (Programming) In `Q6_CNN.ipynb`, implement **`__init__()`** function in `class ThreeLayerConvNet_Pytorch`.
9. (Programming) In `Q6_CNN.ipynb`, implement **`forward()`** function in `class ThreeLayerConvNet_Pytorch`.
11. (Save Outputs) Successfully Run all the **code cells** in this `.ipynb`, $\color{red}{\text{MUST}}$ save the outputs.
12. Submit $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ file to Plato before deadline.


---

## Convolutional Neural Networks Programming Assignment

In the previous programming assignments, neural-network layers were implemented primarily with **NumPy**. The purpose was to understand the mathematical operations behind neural networks, especially the relationship between the **forward pass** and the **backward pass**.

This assignment extends the same idea to **Convolutional Neural Networks (CNNs)**.

The first part of the assignment continues using NumPy. You will manually implement:

* convolution forward pass,
* convolution backward pass,
* max-pooling forward pass,
* max-pooling backward pass,
* and a complete three-layer convolutional neural network.

The three-layer CNN has the following architecture:

$$
\boxed{
X
\rightarrow
\text{Conv}
\rightarrow
\text{ReLU}
\rightarrow
\text{MaxPool}
\rightarrow
\text{Flatten}
\rightarrow
\text{Linear}_1
\rightarrow
\text{ReLU}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Softmax}
\rightarrow
\text{Cross-Entropy}
}
$$

The NumPy implementation is important because it makes the computations inside a convolutional network explicit. In particular, the backward pass shows how gradients are distributed from the output feature map back to the input image, convolutional filters, and biases.

The second part of the assignment introduces **PyTorch**.

Starting from

```python
import torch
```

we move from manually calculating every gradient to using PyTorch's automatic differentiation system, called **Autograd**.

The purpose of this transition is to connect the mathematical understanding developed in the NumPy assignments with the workflow commonly used to train modern deep-learning models.

---

## Setup Code

### Google Colab Setup (Skip If Local Machine)

Unzip the `PA02.zip` file, upload entire `DeepLearning4ComputerVision` folder into Google drivers `My Drive` folder.

Next we need to run a few commands to set up our environment on Google Colab. If you are running this notebook on a local machine you can skip this section.

Run the following cell to mount your Google Drive. Follow the link, sign in to your Google account (the same account you used to store this notebook!) and copy the authorization code into the text box that appears below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'DeepLearning4ComputerVision/PA02'

import os
import sys

#sys.path.append('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)) # if not working, try this line instead
os.chdir('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE))

# Print current working directory
print("Current Working Directory:", os.getcwd())

# List files to make sure your data is visible
print("Available Files:", os.listdir())

If the output shows
> Mounted at /content/drive
> 
> Current Working Directory: /content/drive/My Drive/DeepLearning4ComputerVision/PA02
> 
> Available Files: ['datasets', 'dlcv', 'knn.py', 'softmax.py', ...]

Then, the Google Colab is working correctly.

### python env and ipynb setup (Local Machine Run from here) (Google Colab run this)

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
# Notebook will reload external python modules;
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib.pyplot as plt
# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'


def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

## CIFAR-10 Data Loading and Preprocessing

In [ ]:
from dlcv.data_utils import get_CIFAR10_data
# Load the (preprocessed) CIFAR10 data.
cifar10_dir = 'datasets/cifar-10-batches-py'

data = get_CIFAR10_data(cifar10_dir)
num_train = 200
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}
for k, v in list(small_data.items()):
  print(('%s: ' % k, v.shape))

## Convolutional Neural Networks

A useful mathematical reference for convolutional operation is:

[1] [Bengio, Yoshua, Ian Goodfellow, and Aaron Courville. Deep learning. Vol. 1. Cambridge, MA, USA: MIT press, 2017.](https://www.deeplearningbook.org/)

In particular, Chapter 9 introduces convolutional networks and provides mathematical formulations related to the forward and backward operations of convolution.

The mathematics of convolutional backpropagation can initially be difficult. However, working through the equations and then implementing them directly in NumPy is a valuable exercise. In this assignment, the goal is not only to obtain a correct output from the convolutional layer, but also to trace the gradient all the way from the final loss function back to the first convolutional layer using the chain rule.


### Implement conv forward pass

In `CNN.py`, implement **conv_forward()** function.

In [ ]:
from CNN import conv_forward

x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)
x = np.linspace(-0.1, 0.5, num=np.prod(x_shape)).reshape(x_shape)
w = np.linspace(-0.2, 0.3, num=np.prod(w_shape)).reshape(w_shape)
b = np.linspace(-0.1, 0.2, num=3)

conv_param = {'stride': 2, 'pad': 1}
out, _ = conv_forward(x, w, b, conv_param)
correct_out = np.array([[[[-0.08759809, -0.10987781],
                           [-0.18387192, -0.2109216 ]],
                          [[ 0.21027089,  0.21661097],
                           [ 0.22847626,  0.23004637]],
                          [[ 0.50813986,  0.54309974],
                           [ 0.64082444,  0.67101435]]],
                         [[[-0.98053589, -1.03143541],
                           [-1.19128892, -1.24695841]],
                          [[ 0.69108355,  0.66880383],
                           [ 0.59480972,  0.56776003]],
                          [[ 2.36270298,  2.36904306],
                           [ 2.38090835,  2.38247847]]]])

# Compare your output to ours; difference should be around e-8
print('Testing conv_forward_naive')
print('difference: ', rel_error(out, correct_out))

### Implement conv backward pass

In `CNN.py`, implement **conv_backward()** function.

In [ ]:
from CNN import conv_backward
from dlcv.gradient_check import eval_numerical_gradient_array
np.random.seed(231)
x = np.random.randn(4, 3, 5, 5)
w = np.random.randn(2, 3, 3, 3)
b = np.random.randn(2,)
dout = np.random.randn(4, 2, 5, 5)
conv_param = {'stride': 1, 'pad': 1}

dx_num = eval_numerical_gradient_array(lambda x: conv_forward(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_forward(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_forward(x, w, b, conv_param)[0], b, dout)

out, cache = conv_forward(x, w, b, conv_param)
dx, dw, db = conv_backward(dout, cache)

# Your errors should be around e-8 or less.
print('Testing conv_backward_naive function')
print('dx error: ', rel_error(dx, dx_num))
print('dw error: ', rel_error(dw, dw_num))
print('db error: ', rel_error(db, db_num))

In [ ]:
from imageio import imread
from PIL import Image

kitten = imread('datasets/kitten.jpg')
puppy = imread('datasets/puppy.jpg')
# kitten is wide, and puppy is already square
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # Make this smaller if it runs too slow
resized_puppy = np.array(Image.fromarray(puppy).resize((img_size, img_size)))
resized_kitten = np.array(Image.fromarray(kitten_cropped).resize((img_size, img_size)))
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = resized_puppy.transpose((2, 0, 1))
x[1, :, :, :] = resized_kitten.transpose((2, 0, 1))

# Set up a convolutional weights holding 2 filters, each 3x3
w = np.zeros((2, 3, 3, 3))

# The first filter converts the image to grayscale.
# Set up the red, green, and blue channels of the filter.
w[0, 0, :, :] = [[0, 0, 0], [0, 0.3, 0], [0, 0, 0]]
w[0, 1, :, :] = [[0, 0, 0], [0, 0.6, 0], [0, 0, 0]]
w[0, 2, :, :] = [[0, 0, 0], [0, 0.1, 0], [0, 0, 0]]

# Second filter detects horizontal edges in the blue channel.
w[1, 2, :, :] = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]

# Vector of biases. We don't need any bias for the grayscale
# filter, but for the edge detection filter we want to add 128
# to each output so that nothing is negative.
b = np.array([0, 128])

# Compute the result of convolving each input in x with each filter in w,
# offsetting by b, and storing the results in out.
out, _ = conv_forward(x, w, b, {'stride': 1, 'pad': 1})

def imshow_no_ax(img, normalize=True):
    """ Tiny helper to show images as uint8 and remove axis labels """
    if normalize:
        img_max, img_min = np.max(img), np.min(img)
        img = 255.0 * (img - img_min) / (img_max - img_min)
    plt.imshow(img.astype('uint8'))
    plt.gca().axis('off')

# Show the original images and the results of the conv operation
plt.subplot(2, 3, 1)
imshow_no_ax(puppy, normalize=False)
plt.title('Original image')
plt.subplot(2, 3, 2)
imshow_no_ax(out[0, 0])
plt.title('Grayscale')
plt.subplot(2, 3, 3)
imshow_no_ax(out[0, 1])
plt.title('Edges')
plt.subplot(2, 3, 4)
imshow_no_ax(kitten_cropped, normalize=False)
plt.subplot(2, 3, 5)
imshow_no_ax(out[1, 0])
plt.subplot(2, 3, 6)
imshow_no_ax(out[1, 1])
plt.show()

### Implement Max Pooling forward pass

In `CNN.py`, implement **max_pool_forward()** function.

In [ ]:
from CNN import max_pool_forward
x_shape = (2, 3, 4, 4)
x = np.linspace(-0.3, 0.4, num=np.prod(x_shape)).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

out, _ = max_pool_forward(x, pool_param)

correct_out = np.array([[[[-0.26315789, -0.24842105],
                          [-0.20421053, -0.18947368]],
                         [[-0.14526316, -0.13052632],
                          [-0.08631579, -0.07157895]],
                         [[-0.02736842, -0.01263158],
                          [ 0.03157895,  0.04631579]]],
                        [[[ 0.09052632,  0.10526316],
                          [ 0.14947368,  0.16421053]],
                         [[ 0.20842105,  0.22315789],
                          [ 0.26736842,  0.28210526]],
                         [[ 0.32631579,  0.34105263],
                          [ 0.38526316,  0.4       ]]]])

# Compare your output with ours. Difference should be on the order of e-8.
print('Testing max_pool_forward_naive function:')
print('difference: ', rel_error(out, correct_out))

### Implement Max Pooling backward pass

In `CNN.py`, implement **max_pool_backward()** function.

In [ ]:
from CNN import max_pool_backward
np.random.seed(231)
x = np.random.randn(3, 2, 8, 8)
dout = np.random.randn(3, 2, 4, 4)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

dx_num = eval_numerical_gradient_array(lambda x: max_pool_forward(x, pool_param)[0], x, dout)

out, cache = max_pool_forward(x, pool_param)
dx = max_pool_backward(dout, cache)

# Your error should be on the order of e-12
print('Testing max_pool_backward_naive function:')
print('dx error: ', rel_error(dx, dx_num))

### Implement Three-Layer Conv Net

In `three_layer_conv_net.py`, implement **`__init__()`** function in `class ThreeLayerConvNet`.

In `three_layer_conv_net.py`, implement **loss()** function in `class ThreeLayerConvNet`.

For the Three-Layer Covn Net model:

$$
\boxed{
X
\rightarrow
\text{Conv}
\rightarrow
\text{Relu}
\rightarrow
\text{MaxPool}
\rightarrow
\text{Flatten}
\rightarrow
\text{Linear}_1
\rightarrow
\text{Relu}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Softmax}
\rightarrow
\text{Cross-Entropy}.
}
$$

In [ ]:
from three_layer_conv_net import ThreeLayerConvNet
from dlcv.gradient_check import eval_numerical_gradient
num_inputs = 2
input_dim = (3, 16, 16)
reg = 0.0
num_classes = 10
np.random.seed(231)
X = np.random.randn(num_inputs, *input_dim)
y = np.random.randint(num_classes, size=num_inputs)

model = ThreeLayerConvNet(
    num_filters=3,
    filter_size=3,
    input_dim=input_dim,
    hidden_dim=7,
    dtype=np.float64
)
loss, grads = model.loss(X, y)
# Errors should be small, but correct implementations may have
# relative errors up to the order of e-2
for param_name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    param_grad_num = eval_numerical_gradient(f, model.params[param_name], verbose=False, h=1e-6)
    e = rel_error(param_grad_num, grads[param_name])
    print('%s max relative error: %e' % (param_name, rel_error(param_grad_num, grads[param_name])))

### Overfit Small Data
A nice trick is to train your model with just a few training samples. You should be able to overfit small datasets, which will result in very high training accuracy and comparatively low validation accuracy.

In [ ]:
from dlcv.solver import Solver
np.random.seed(231)

model = ThreeLayerConvNet()

solver = Solver(
    model,
    small_data,
    num_epochs=15,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=1
)
solver.train()

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(solver.loss_history, 'o')
plt.xlabel('iteration')
plt.ylabel('loss')

plt.subplot(2, 1, 2)
plt.plot(solver.train_acc_history, '-o')
plt.plot(solver.val_acc_history, '-o')
plt.legend(['train', 'val'], loc='upper left')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

## Pytorch

### Transition from NumPy to PyTorch

Up to this point, the assignments have mainly used NumPy.

Starting from this section,


we begin using **PyTorch**.

PyTorch provides an automatic differentiation system called **Autograd**.

The major difference is:

$$
\boxed{
\text{NumPy: forward and backward are implemented manually}
}
$$

whereas

$$
\boxed{
\text{PyTorch: define the forward computation, and Autograd computes the backward gradients}
}
$$

for tensors whose gradients are being tracked.


### Using CUDA in PyTorch

One important advantage of PyTorch is its ability to perform neural-network computations on a GPU using CUDA.

CUDA is NVIDIA's platform for performing general-purpose computation on NVIDIA GPUs. Deep-learning workloads contain many large matrix and tensor operations that can be executed in parallel. For example, convolution, matrix multiplication, and gradient computation repeatedly perform large numbers of multiplication and addition operations.

Because GPUs contain many parallel processing units, they can perform these operations much faster than a CPU for sufficiently large neural networks and datasets.



In [ ]:
import torch
USE_GPU = True
if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('using device:', device)

### Autograd

This example introduces the basic PyTorch training workflow using a simple linear model.

The key concept is Autograd. When trainable parameters are created with gradient tracking enabled, PyTorch records the operations used during the forward pass. After the loss is computed, calling loss.backward() automatically applies the chain rule and calculates the gradients for all trainable parameters.



In [ ]:
# --------------------------------
# Toy dataset: y = 2x + 1
# --------------------------------
x = torch.tensor([[1.0],
                  [2.0]])

y = torch.tensor([[3.0],
                  [5.0]])

# --------------------------------
# Parameters
# requires_grad=True tells PyTorch
# to track operations for autograd
# --------------------------------
w = torch.tensor([[0.0]], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

learning_rate = 0.1

# --------------------------------
# Training loop
# --------------------------------
for epoch in range(20):

    # Forward
    y_pred = x @ w + b

    # Mean squared error
    loss = ((y_pred - y) ** 2).mean()

    # Backward
    # Computes:
    #   dLoss/dw -> w.grad
    #   dLoss/db -> b.grad
    loss.backward()

    # Print gradients
    print(
        f"Epoch {epoch:2d} | "
        f"Loss: {loss.item():.4f} | "
        f"w: {w.item():.4f} | "
        f"b: {b.item():.4f} | "
        f"dw: {w.grad.item():.4f} | "
        f"db: {b.grad.item():.4f}"
    )

    # --------------------------------
    # Gradient descent update
    # --------------------------------
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # --------------------------------
    # Clear gradients
    # PyTorch accumulates gradients
    # by default
    # --------------------------------
    w.grad.zero_()
    b.grad.zero_()

print("\nFinal:")
print("w =", w.item())
print("b =", b.item())

### Loading datasets with Pytorch API

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data
# and for performing data augmentation; here we set up a transform to
# preprocess the data by subtracting the mean RGB value and dividing by the
# standard deviation of each RGB value; we've hardcoded the mean and std.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test); Datasets load
# training examples one at a time, so we wrap each Dataset in a DataLoader which
# iterates through the Dataset and forms minibatches. We divide the CIFAR-10
# training set into train and val sets by passing a Sampler object to the
# DataLoader telling how it should sample from the underlying Dataset.
cifar10_train = dset.CIFAR10('datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

### Implement Three-Layer Conv Net with Pytorch

For the Three-Layer Covn Net model:

$$
\boxed{
X
\rightarrow
\text{Conv}
\rightarrow
\text{Relu}
\rightarrow
\text{MaxPool}
\rightarrow
\text{Flatten}
\rightarrow
\text{Linear}_1
\rightarrow
\text{Relu}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Output}_{logits}
}
$$

In [ ]:
import torch.nn as nn
import torch.nn.functional as F  # useful stateless functions


class ThreeLayerConvNet_Pytorch(nn.Module):
    def __init__(self, in_channel=3, out_channel=24, hidden_dim=100, num_classes=10):
        super().__init__()
        ########################################################################
        # TODO: Set up the layers for a three-layer ConvNet with the following #
        # architecture:                                                       #
        #                                                                      #
        #   Conv -> ReLU -> MaxPool -> Flatten -> Linear -> ReLU -> Linear    #
        #                                                                      #
        # In this function, define only the trainable / parameterized layers.  #
        # Hint: use                                                      #
        #   - nn.Conv2d for the convolution layer                              #
        #   - nn.MaxPool2d for the max-pooling layer                           #
        #   - nn.Linear for the two fully-connected layers                     #
        #                                                                      #
        # The input image size is 32 x 32.                                     #
        ########################################################################

        pass
        self.conv1 = nn.Conv2d(
            in_channel,
            out_channel,
            kernel_size=7,
            padding=3
        )

        # 32x32 -> 16x16
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Flatten: channel_1 * 16 * 16
        self.fc1 = nn.Linear(
            out_channel * 16 * 16,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            num_classes
        )

        nn.init.kaiming_normal_(self.conv1.weight)
        nn.init.kaiming_normal_(self.fc1.weight)
        nn.init.kaiming_normal_(self.fc2.weight)
        # Replace "pass" statement with your code
        pass
        ########################################################################
        #                          END OF YOUR CODE                            #
        ########################################################################

    def forward(self, x):
        scores = None

        ########################################################################
        # TODO: Implement the forward pass using the layers defined in         #
        # __init__. The network should follow this structure:                  #
        #                                                                      #
        #   Conv -> ReLU -> MaxPool -> Flatten -> Linear -> ReLU -> Linear    #
        #                                                                      #
        # Hint: use                                        #
        #   - the nn.Conv2d layer defined in __init__                          #
        #   - F.relu for both ReLU activations                                 #
        #   - the nn.MaxPool2d layer defined in __init__                       #
        #   - torch.flatten(x, start_dim=1) to flatten each sample             #
        #   - the two nn.Linear layers defined in __init__                     #
        #                                                                      #
        # The final output, scores, should have shape:                         #
        #                                                                      #
        #   [batch_size, num_classes]                                          #
        # Don't apply softmax to scores.
        ########################################################################

        pass
        # Conv
        x = self.conv1(x)

        # ReLU
        x = F.relu(x)

        # MaxPool
        x = self.pool(x)

        # Flatten
        x = torch.flatten(x, start_dim=1)

        # Linear
        x = self.fc1(x)

        # ReLU
        x = F.relu(x)

        # Linear
        scores = self.fc2(x)

        # Replace "pass" statement with your code
        pass
        ########################################################################
        #                             END OF YOUR CODE                         #
        ########################################################################

        return scores


x = torch.zeros((64, 3, 32, 32))  # minibatch size 64, image size [3, 32, 32]
model = ThreeLayerConvNet_Pytorch(in_channel=3, out_channel=12, hidden_dim=8, num_classes=10)
scores = model(x)
print(scores.size())  # you should see [64, 10]


### Training Model

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Use device:", device)

model = ThreeLayerConvNet_Pytorch().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# if no cuda, then epoch=2
num_epochs = 5 if device.type=="cuda" else 2

for epoch in range(num_epochs):    
    # Train    
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, y in loader_train:
        x = x.to(device)
        y = y.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward
        scores = model(x)

        # Loss
        loss = criterion(scores, y)

        # Backward
        loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate loss
        train_loss += loss.item() * x.size(0)

        # Accuracy
        pred = scores.argmax(dim=1)

        train_correct += (pred == y).sum().item()
        train_total += y.size(0)        

    train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    
    # Test    
    model.eval()

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for x, y in loader_test:
            x = x.to(device)
            y = y.to(device)

            # Forward
            scores = model(x)

            # Loss
            loss = criterion(scores, y)

            test_loss += loss.item() * x.size(0)

            # Accuracy
            pred = scores.argmax(dim=1)

            test_correct += (pred == y).sum().item()
            test_total += y.size(0)

    test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    
    # Print results    
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc * 100:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc * 100:.2f}%"
    )
   

### Why in Pytorch logits → CrossEntropyLoss?

*(In here, the scores is the logtis)*

In teaching, multiclass classification is usually explained as:

logits → softmax → cross-entropy loss

Softmax first converts the logits into class probabilities, and cross-entropy then measures how well those probabilities match the correct label.

In PyTorch, however, `CrossEntropyLoss` takes the **raw logits directly**. This is because PyTorch combines the softmax-related computation and cross-entropy into a single numerically stable operation.

Conceptually:

logits → softmax → cross-entropy

but in PyTorch:

logits → `CrossEntropyLoss`

Internally, PyTorch uses `log_softmax` together with negative log-likelihood. This is mathematically equivalent to applying softmax followed by cross-entropy, but it is more numerically stable and efficient.

Therefore, when using `CrossEntropyLoss`, the model should output logits directly, and `softmax` should not be applied beforehand.
